In [5]:
# Importing libraries
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Function to train and evaluate KNN
def run_knn(df, target_column):
    X = df.drop(columns=[target_column])
    y = df[target_column]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)

    print("Accuracy:", round(accuracy_score(y_test, y_pred), 2))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))


# TITANIC
titanic = pd.read_csv('titanic.csv')
titanic.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)
titanic['Age'].fillna(titanic['Age'].median(), inplace=True)
titanic['Embarked'].fillna(titanic['Embarked'].mode()[0], inplace=True)
titanic['Sex'] = titanic['Sex'].map({'male': 0, 'female': 1})
titanic['Embarked'] = titanic['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

print("Titanic Dataset")
run_knn(titanic, 'Survived')


# DIABETES
diabetes = pd.read_csv('diabetes_dataset.csv')
cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in cols:
    diabetes[col] = diabetes[col].replace(0, diabetes[col].median())

print("\nDiabetes Dataset")
run_knn(diabetes, 'Outcome')


# SOCIAL NETWORK ADS
ads = pd.read_csv('Social_Network_Ads.csv')
ads.drop(columns=['User ID'], inplace=True)
ads['Gender'] = ads['Gender'].map({'Male': 0, 'Female': 1})

print("\nSocial Network Ads Dataset")
run_knn(ads, 'Purchased')

Titanic Dataset
Accuracy: 0.81
Confusion Matrix:
 [[91 14]
 [20 54]]
Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.87      0.84       105
           1       0.79      0.73      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179


Diabetes Dataset
Accuracy: 0.75
Confusion Matrix:
 [[80 19]
 [19 36]]
Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.81      0.81        99
           1       0.65      0.65      0.65        55

    accuracy                           0.75       154
   macro avg       0.73      0.73      0.73       154
weighted avg       0.75      0.75      0.75       154


Social Network Ads Dataset
Accuracy: 0.92
Confusion Matrix:
 [[48  4]
 [ 2 26]]
Classification Report:
               precision    recall  f1-score   support

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler   

k = 5

#  KNN FUNCTIONS 
def predict_knn(X_train, y_train, x_test):
    dist = [(np.sqrt(np.sum((x - x_test) ** 2)), y_train[i]) for i, x in enumerate(X_train)]
    dist.sort(key=lambda x: x[0])
    labels = [label for _, label in dist[:k]]
    return max(set(labels), key=labels.count)

def accuracy(y_true, y_pred):
    return sum(y_true[i] == y_pred[i] for i in range(len(y_true))) / len(y_true)

def split(X, y):
    s = int(0.8 * len(X))
    return X[:s], X[s:], y[:s], y[s:]


# LOAD DATA 
df_diabetes = pd.read_csv('diabetes_dataset.csv')
df_social = pd.read_csv('Social_Network_Ads.csv')
df_titanic = pd.read_csv('titanic.csv')

print("Diabetes\n", df_diabetes.head(), "\nMissing:\n", df_diabetes.isnull().sum())
print("\nSocial Network Ads\n", df_social.head(), "\nMissing:\n", df_social.isnull().sum())
print("\nTitanic\n", df_titanic.head(), "\nMissing:\n", df_titanic.isnull().sum())


#  PREPROCESS 
df_titanic['Age'].fillna(df_titanic['Age'].median(), inplace=True)
df_titanic['Embarked'].fillna(df_titanic['Embarked'].mode()[0], inplace=True)
df_titanic.drop(columns=['Cabin','PassengerId','Name','Ticket'], inplace=True)

for col in ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']:
    df_diabetes[col] = df_diabetes[col].replace(0, df_diabetes[col].median())

df_social['Gender'] = df_social['Gender'].map({'Male':0,'Female':1})
df_titanic['Sex'] = df_titanic['Sex'].map({'male':0,'female':1})
df_titanic['Embarked'] = df_titanic['Embarked'].map({'S':0,'C':1,'Q':2})


#  PREPARE
X_d, y_d = df_diabetes.drop('Outcome', axis=1).values, df_diabetes['Outcome'].values
X_s, y_s = df_social.drop(['User ID','Purchased'], axis=1).values, df_social['Purchased'].values
X_t, y_t = df_titanic.drop('Survived', axis=1).values, df_titanic['Survived'].values


# FEATURE SCALING 
scaler = StandardScaler()  

X_d = scaler.fit_transform(X_d)   
X_s = scaler.fit_transform(X_s)   
X_t = scaler.fit_transform(X_t)   


# SPLIT 
datasets = [
    ("Diabetes", *split(X_d, y_d)),
    ("Social Ads", *split(X_s, y_s)),
    ("Titanic", *split(X_t, y_t))
]


# TRAIN & TEST 
for name, X_tr, X_te, y_tr, y_te in datasets:
    print(f"\n{name} Results")
    preds = [predict_knn(X_tr, y_tr, x) for x in X_te]
    print("Accuracy:", round(accuracy(y_te, preds), 4))

Diabetes
    Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1   
Missing:
 Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64